In [1]:
import torch

In [2]:
def make_causal_mask(seq_len):
  temp = torch.ones(seq_len,seq_len)
  tri = torch.tril(temp,diagonal=0)

  return tri.bool()

In [3]:
def make_padding_mask(token_ids, pad_id=0):
  mask = token_ids.ne(pad_id)
  mask = mask.unsqueeze(1)

  return mask

In [5]:
def masked_attention(q, k, v, mask=None):
  dk = (q.size(-1)) ** 0.5
  score = q @ k.transpose(-2,-1)
  scores = score / dk

  if mask is not None:
    scores = scores.masked_fill(~mask,float('-inf'))
  weights = torch.softmax(scores,dim=-1)
  output = weights @ v
  return output,weights

In [7]:
torch.manual_seed(42)

B = 2   # Batch size (样本数)
T = 4   # Sequence length (序列长度)
d = 8   # 特征维度 (d_k)

q = torch.randn(B, T, d)
k = torch.randn(B, T, d)
v = torch.randn(B, T, d)

token_ids = torch.tensor([
    [5, 7, 9, 0],
    [2, 8, 0, 0]
])

casual_mask = make_causal_mask(T)
padding_mask = make_padding_mask(token_ids)
combained_mask = casual_mask & padding_mask

print("--- 检查 Mask 形状 ---")
print("causal_mask shape:", casual_mask.shape)
print("padding_mask shape:", padding_mask.shape)
print("combined_mask shape:", combained_mask.shape)


out,weights = masked_attention(q,k,v,mask=None)
print("\n--- 场景 A: 无 Mask ---")
print("Output shape:", out.shape)      # 预期: [2, 4, 8]
print("Weights shape:", weights.shape) # 预期: [2, 4, 4]
print("第一行权重和 (应该为 1):", weights[0, 0].sum().item())


out_masked,weights_masked = masked_attention(q, k, v, mask=combained_mask)
print("\n--- 场景 B: 组合 Mask (Causal + Padding) ---")
print("Output shape:", out_masked.shape)
print("Weights shape:", weights_masked.shape)

# 打印第一个样本的权重矩阵，观察被 mask 的情况
print("\n样本 1 的 Attention Weights 矩阵 (观察 0 的位置):")
# 使用 .numpy() 或者直接 print 看看被 mask 的地方是不是变成了 0
print(weights_masked[0].detach().numpy().round(decimals=3))

print("\n样本 2 的 Attention Weights 矩阵 (观察 0 的位置):")
print(weights_masked[1].detach().numpy().round(decimals=3))

--- 检查 Mask 形状 ---
causal_mask shape: torch.Size([4, 4])
padding_mask shape: torch.Size([2, 1, 4])
combined_mask shape: torch.Size([2, 4, 4])

--- 场景 A: 无 Mask ---
Output shape: torch.Size([2, 4, 8])
Weights shape: torch.Size([2, 4, 4])
第一行权重和 (应该为 1): 1.0

--- 场景 B: 组合 Mask (Causal + Padding) ---
Output shape: torch.Size([2, 4, 8])
Weights shape: torch.Size([2, 4, 4])

样本 1 的 Attention Weights 矩阵 (观察 0 的位置):
[[1.    0.    0.    0.   ]
 [0.406 0.594 0.    0.   ]
 [0.52  0.051 0.428 0.   ]
 [0.84  0.121 0.039 0.   ]]

样本 2 的 Attention Weights 矩阵 (观察 0 的位置):
[[1.    0.    0.    0.   ]
 [0.493 0.507 0.    0.   ]
 [0.543 0.457 0.    0.   ]
 [0.531 0.469 0.    0.   ]]
